<a href="https://colab.research.google.com/github/reemacodess/MATH319-GroupProject/blob/main/Part1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sqlite3, hashlib, hmac, secrets, os
from datetime import datetime, timezone
import ipywidgets as widgets
from IPython.display import display, clear_output

DB_PATH = 'voting.db'  # Colab temporary storage; export if needed
CANDIDATES = {'A': 'Candidate A', 'B': 'Candidate B', 'C': 'Candidate C'}

def hash_password(password, salt=None):
    salt = salt or os.urandom(16)
    derived = hashlib.pbkdf2_hmac('sha256', password.encode('utf-8'), salt, 200_000)
    return salt.hex(), derived.hex()

def verify_password(password, salt_hex, expected_hex):
    _, result = hash_password(password, bytes.fromhex(salt_hex))
    return hmac.compare_digest(result, expected_hex)

def connection():
    con = sqlite3.connect(DB_PATH)
    con.execute('PRAGMA foreign_keys = ON')
    return con

def initialize_students():
    with connection() as con:
        con.execute("""CREATE TABLE IF NOT EXISTS students (
            student_id TEXT PRIMARY KEY,
            password_salt TEXT NOT NULL,
            password_hash TEXT NOT NULL,
            eligible INTEGER NOT NULL CHECK(eligible IN (0,1))
        )""")

def add_demo_student(student_id, password, eligible):
    salt, digest = hash_password(password)
    with connection() as con:
        con.execute('INSERT OR IGNORE INTO students VALUES (?, ?, ?, ?)',
                    (student_id, salt, digest, int(eligible)))

def authenticate(student_id, password):
    with connection() as con:
        row = con.execute('SELECT password_salt, password_hash, eligible FROM students WHERE student_id=?',
                          (student_id,)).fetchone()
    if row is None or not verify_password(password, row[0], row[1]):
        return False, 'Invalid student ID or password.'
    if not row[2]:
        return False, 'This student is not eligible to vote.'
    return True, 'Login successful. Student is eligible.'

initialize_students()
add_demo_student('student01', 'DemoPass01!', True)
add_demo_student('student02', 'DemoPass02!', True)
add_demo_student('student03', 'DemoPass03!', False)
print("Database and demo student accounts are ready.")

Database and demo student accounts are ready.


In [ ]:
student_input = widgets.Text(description='Student ID:')
password_input = widgets.Password(description='Password:')
login_button = widgets.Button(description='Log in', button_style='primary')
login_output = widgets.Output()
candidate_select = widgets.RadioButtons(options=[(name, key) for key, name in CANDIDATES.items()], description='Candidate:')
confirm_button = widgets.Button(description='Confirm selection', button_style='success')
selection_output = widgets.Output()
selection_panel = widgets.VBox([widgets.HTML('<h3>Select one candidate</h3>'), candidate_select, confirm_button, selection_output])
selection_panel.layout.display = 'none'
state = {'student_id': None, 'pending_candidate': None}

# Backend integration point. Teammates register their function after loading this notebook.
# Function signature: handler(student_id: str, candidate_id: str) -> (bool, str)
# The handler must handle token issuance/validation, deadline, encryption,
# atomic vote storage and token consumption. It must not store identity with votes.
_vote_handler = None

def register_vote_handler(handler):
    global _vote_handler
    if not callable(handler):
        raise TypeError('handler must be callable')
    _vote_handler = handler

def submit_vote(student_id, candidate_id):
    if candidate_id not in CANDIDATES:
        return False, 'Invalid candidate.'
    if _vote_handler is None:
        return False, 'Voting backend is not connected yet. No vote was stored.'
    try:
        success, message = _vote_handler(student_id, candidate_id)
        return bool(success), str(message)
    except Exception:
        return False, 'Vote could not be submitted. Please contact the administrator.'


def on_login(_):
    ok, message = authenticate(student_input.value.strip(), password_input.value)
    state['student_id'] = student_input.value.strip() if ok else None
    state['pending_candidate'] = None
    confirm_button.disabled = False
    selection_panel.layout.display = '' if ok else 'none'
    with login_output:
        clear_output()
        print(message)
    with selection_output:
        clear_output()

def on_confirm(_):
    with selection_output:
        clear_output()
        if not state['student_id']:
            print('Please log in first.')
            return
        selected = candidate_select.value
        state['pending_candidate'] = selected
        success, message = submit_vote(state['student_id'], selected)
        print(message)
        if success:
            confirm_button.disabled = True

login_button.on_click(on_login)
confirm_button.on_click(on_confirm)
display(widgets.VBox([widgets.HTML('<h2>Secure Classroom Voting — Student Portal</h2>'),
                      student_input, password_input, login_button, login_output, selection_panel]))

In [ ]:
assert authenticate('student01', 'DemoPass01!')[0] is True
assert authenticate('student01', 'wrong-password')[0] is False
assert authenticate('unknown', 'anything')[0] is False
assert authenticate('student03', 'DemoPass03!')[0] is False
assert len(CANDIDATES) == 3
with connection() as con:
    tables = [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table'")]
    columns = [r[1] for r in con.execute('PRAGMA table_info(students)')]
assert 'students' in tables
# Other tables may be added when teammates integrate their components.
assert 'password_hash' in columns and 'password_salt' in columns
print('PASS: valid login, invalid password, unknown student, ineligible student,')
print('candidate options, student-only database schema and hashed credentials.')
assert submit_vote('student01', 'B')[0] is False


PASS: valid login, invalid password, unknown student, ineligible student,
candidate options, student-only database schema and hashed credentials.
